# Canary-Qwen Batch Evaluation

This notebook evaluates the NVIDIA Canary-Qwen-2.5B model on a dataset of audio segments.
It uses the NeMo framework directly for batch inference.

In [ ]:
import os
import json
import torch
from google.cloud import storage
from nemo.collections.speechlm2.models import SALM

# Import common GCS utils and runner (CORRECTED)
from common.gcs_utils import parse_gcs_uri, download_blob_to_file, download_jsonl_manifest, upload_inference_results
from common.eval_runner import run_batch_evaluation

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Loading NVIDIA Canary-Qwen-2.5B into GPU VRAM...")
model = SALM.from_pretrained('nvidia/canary-qwen-2.5b').half().eval().to("cuda")

In [ ]:
# Configuration
GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>" 
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"
MODEL_NAME = "<MODEL_NAME>"
BATCH_SIZE = 4  # Smaller batch size for safety
LIMIT = 10  # For testing

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Define prompt formatter for Canary-Qwen
def prompt_formatter(entry, local_path):
    return [
        {
            "role": "user",
            "content": "Transcribe the audio accurately.\n\n" + model.audio_locator_tag,
            "audio": [local_path]
        }
    ]

# Define inference function for Canary-Qwen (SALM)
def canary_inference(model, prompts):
    return model.generate(prompts=prompts, max_new_tokens=256)

# Define result decoder for Canary-Qwen
def result_decoder(ans, model):
    if isinstance(ans, str):
        return ans
    else:
        return model.tokenizer.ids_to_text(ans.cpu())

# Run the generic batch evaluation
results_list = run_batch_evaluation(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=canary_inference,
    decode_fn=result_decoder,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=MODEL_NAME,
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

In [ ]:
# Upload results directly to GCS from memory (uncomment to use)
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    MODEL_NAME, 
    EXPERIMENT_NAME, 
    results_list
)